<a href="https://colab.research.google.com/github/DuhranDuhran/AI-tool-calling/blob/main/Guardrails_and_Error_Handling_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Guardrails and Error Handling

In [2]:
import re
from datetime import datetime
from google import genai
from google.genai import types, errors
from google.colab import userdata

# Initialize Client
api_key = userdata.get("Gemini_API_Key1")
client = genai.Client(api_key=api_key)


# Tool Function with Multi-Layered Parameter Guardrails
def schedule_consultation(client_email: str, booking_date: str, hours: int) -> str:
    """Schedules an enterprise consultation after validating input parameters.

    Args:
        client_email: Client email address (must follow valid name@domain.com format).
        booking_date: Desired consultation date formatted strictly as YYYY-MM-DD.
        hours: Consultation duration in hours (must be between 1 and 8).
    """
    # Guardrail 1: Regex Email Format Check
    email_pattern = r"^[\w\.-]+@[\w\.-]+\.\w+$"
    if not re.match(email_pattern, client_email):
        return f"Error: '{client_email}' is not a valid email address format."

    # Guardrail 2: Strict Date Parsing
    try:
        parsed_date = datetime.strptime(booking_date, "%Y-%m-%d")
    except ValueError:
        return f"Error: Date '{booking_date}' must follow strict YYYY-MM-DD format."

    # Guardrail 3: Numeric Range Bounds
    if hours < 1 or hours > 8:
        return f"Error: Requested duration ({hours} hrs) is out of bounds. Must be 1–8 hours."

    return f"Success: Scheduled a {hours}-hour consultation for {client_email} on {parsed_date.strftime('%B %d, %Y')}."


# Execution Wrapper with API-Level Exception Catching
def execute_agent_prompt(prompt: str):
    print(f"\nUser Query: {prompt}")
    try:
        response = client.models.generate_content(
            model="gemini-3.6-flash",
            contents=prompt,
            config=types.GenerateContentConfig(tools=[schedule_consultation])
        )
        print(f"Gemini Response:\n{response.text}")

    except errors.APIError as e:
        print(f"API Error ({e.code}): {e.message}")
    except Exception as e:
        print(f"System Error: {str(e)}")


# --- TEST CASES ---
# Test 1: Bad Email Format
execute_agent_prompt("Book 2 hours for bad-email-address on 2026-11-05")

# Test 2: Bad Date Format
execute_agent_prompt("Book 3 hours for dev@example.com on November 5th 2026")

# Test 3: Valid Execution
execute_agent_prompt("Schedule a 4 hour consultation for dev@example.com on 2026-11-05")


User Query: Book 2 hours for bad-email-address on 2026-11-05
Gemini Response:
The consultation could not be scheduled because `bad-email-address` is not a valid email address. Please provide a valid email address (e.g., `name@domain.com`) to proceed with the booking.

User Query: Book 3 hours for dev@example.com on November 5th 2026
Gemini Response:
The 3-hour enterprise consultation for dev@example.com has been successfully scheduled for November 5, 2026.

User Query: Schedule a 4 hour consultation for dev@example.com on 2026-11-05
Gemini Response:
A 4-hour enterprise consultation has been successfully scheduled for dev@example.com on November 5, 2026.
